

```
# https://drive.google.com/file/d/19B-yNR1tOfydDucWwHfyVGAZ8fEbhDIR/view?usp=sharing
```



In [1]:
# 1. Install Unsloth natively via their official pre-compiled GitHub wheel for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install essential dependencies without dependency-resolution lockups
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# 3. Install utility libraries for Kaggle integration and dataset management
!pip install kagglehub datasets transformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-zhtq9_33/unsloth_c9f716b1e53c4a3080191b88479b49fe
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-zhtq9_33/unsloth_c9f716b1e53c4a3080191b88479b49fe
  Resolved https://github.com/unslothai/unsloth.git to commit aba57d0872d75e167d375db004b4d64f676bd11f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.1 MB/s eta 0:00:00


In [2]:
# ==========================================
# 0. DEEP VRAM FLUSH & MEMORY PURGE
# ==========================================
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 1. ENVIRONMENT SETUP & DATA IMPORT
# ==========================================
import kagglehub
import os
import json
from datasets import Dataset
from transformers import EarlyStoppingCallback, TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# ==========================================
# 2. ROBUST DATA LOADING
# ==========================================
def load_and_format_dataset(file_path):
    formatted_data = []
    line_count = 0
    skipped_count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        first_char = f.read(1)
        f.seek(0)

        if first_char == '[':
            records = json.load(f)
        else:
            records = []
            for line in f:
                line_count += 1
                cleaned_line = line.strip()
                if not cleaned_line: continue
                try:
                    records.append(json.loads(cleaned_line))
                except json.JSONDecodeError:
                    skipped_count += 1
                    continue

    for record in records:
        jd = record.get("job_description", "")
        # Core change: targeting 'keyword_profile' instead of evaluation questions
        keyword_profile = record.get("keyword_profile", {})

        if not jd or not keyword_profile: continue

        # Format the expected assistant response as a clean JSON string
        target_output = json.dumps(keyword_profile, ensure_ascii=False)

        messages = [
            {
                "role": "system",
                "content": "You are a professional HR data extraction assistant. Analyze the provided Job Description and extract the key terms into a structured JSON profile containing 'primary', 'secondary', and 'tertiary' keyword arrays. Output ONLY the raw JSON block without formatting wrappers or commentary."
            },
            {
                "role": "user",
                "content": f"Extract the keyword profile from this job description:\n\n{jd}"
            },
            {
                "role": "assistant",
                "content": target_output
            }
        ]
        formatted_data.append({"messages": messages})

    print(f"Successfully loaded {len(formatted_data)} records. (Skipped: {skipped_count} rows)")
    return Dataset.from_list(formatted_data)

def apply_formatting_template(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

In [4]:
INPUT_FILE = "/content/era_match_keywords.jsonl"
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct" # Or "unsloth/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "outputs/jd_to_keywords_model"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True

raw_dataset = load_and_format_dataset(INPUT_FILE)

dataset_split = raw_dataset.train_test_split(test_size=0.05, seed=3407)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

template_type = "llama-3.1" if "llama" in MODEL_NAME.lower() else ("qwen-2.5" if "qwen" in MODEL_NAME.lower() else "phi-3.5")
tokenizer = get_chat_template(tokenizer, chat_template = template_type)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = train_dataset.map(apply_formatting_template, batched = True)
eval_dataset = eval_dataset.map(apply_formatting_template, batched = True)

Successfully loaded 5014 records. (Skipped: 0 rows)
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/4763 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

In [5]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    per_device_eval_batch_size = 2,
    eval_accumulation_steps = 1,
    warmup_steps = 10,

    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 10,

    eval_strategy = "steps",
    eval_steps = 100,
    save_strategy = "steps",
    save_steps = 100,
    save_total_limit = 2,
    load_best_model_at_end = False,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,

    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    optim = "adamw_8bit",
    weight_decay = 0.01,
    seed = 3407,
    output_dir = "outputs",

    # --- MOVE MAX_SEQ_LENGTH HERE ---
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_text_field = "text", # Safely handles the text field mapping here too
)

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = args, # SFTConfig passes everything down automatically
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 2)]
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4763 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/251 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [6]:
trainer_stats = trainer.train()

best_checkpoint_path = trainer.state.best_model_checkpoint

if best_checkpoint_path is not None:
    model.from_pretrained(model, best_checkpoint_path)
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
else:
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,763 | Num Epochs = 1 | Total steps = 596
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.497013,0.474066
200,0.446488,0.449762
300,0.443031,0.436575
400,0.422356,0.428732
500,0.418871,0.422721
596,0.420100,0.420125


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.

In [7]:
!zip -r "keywords_model.zip" "outputs/jd_to_keywords_model"

  adding: outputs/jd_to_keywords_model/ (stored 0%)
  adding: outputs/jd_to_keywords_model/adapter_config.json (deflated 59%)
  adding: outputs/jd_to_keywords_model/chat_template.jinja (deflated 72%)
  adding: outputs/jd_to_keywords_model/README.md (deflated 65%)
  adding: outputs/jd_to_keywords_model/tokenizer_config.json (deflated 96%)
  adding: outputs/jd_to_keywords_model/adapter_model.safetensors (deflated 72%)
  adding: outputs/jd_to_keywords_model/tokenizer.json (deflated 85%)


In [8]:
# ==========================================
# 5. INFERENCE VERIFICATION
# ==========================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "outputs/jd_to_keywords_model",
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

test_jd = """
We are looking for a Senior DevOps Engineer with 5+ years of experience.
Must have strong expertise in Kubernetes cluster administration, Terraform for AWS Infrastructure,
and building Gitlab CI/CD pipelines. Experience with Prometheus and Grafana is a plus.
"""

messages = [
    {
        "role": "system",
        "content": "You are a professional HR data extraction assistant. Analyze the provided Job Description and extract the key terms into a structured JSON profile containing 'primary', 'secondary', and 'tertiary' keyword arrays. Output ONLY the raw JSON block without formatting wrappers or commentary."
    },
    {
        "role": "user",
        "content": f"Extract the keyword profile from this job description:\n\n{test_jd}"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1, # Dropped temperature to keep JSON structured and deterministic
    top_p = 0.9
)

# Decode response starting right after our input tokens
generated_output = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("Extracted Profile Output:\n", generated_output)

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_m

Extracted Profile Output:
 {"primary":["Senior DevOps Engineer","Kubernetes cluster administration","Terraform","AWS Infrastructure","Gitlab CI/CD","Prometheus","Grafana"],"secondary":[],"tertiary":[]}


### **Evaluation Pillar**

In [9]:
!pip install evaluate rouge_score bleu bert_score

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=99b0f0abf48026637cb58d6c7d545e1d68229e164c38e6fd961448e09d15718e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
  Created wheel for bleu: filename=bleu-0.3-py3-none-any.whl size=5780 sha256=4deb8098746c330c4dc0ce06734f88a87d9f20f09f3c3d61080f306ac7f88069
  Stored in directory: /root/.cache/pip/wheels/a0/08/b2/15cf219d07cdbbc5b1be2d863de6c87e04f99274098e22d061
Successfully built rouge_score bleu


In [ ]:
import os
import torch
import evaluate
from tqdm import tqdm
from unsloth import FastLanguageModel

FINETUNED_MODEL_PATH = "outputs/jd_to_keywords_model"

print("Loading fine-tuned Unsloth model for evaluation...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = FINETUNED_MODEL_PATH,
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\n--- Running Quick Manual Verification ---")
test_jd = """
We are looking for a Senior DevOps Engineer with 5+ years of experience.
Must have strong expertise in Kubernetes cluster administration, Terraform for AWS Infrastructure,
and building Gitlab CI/CD pipelines. Experience with Prometheus and Grafana is a plus.
"""

messages = [
    {
        "role": "system",
        "content": "You are a professional HR data extraction assistant. Analyze the provided Job Description and extract the key terms into a structured JSON profile containing 'primary', 'secondary', and 'tertiary' keyword arrays. Output ONLY the raw JSON block without formatting wrappers or commentary."
    },
    {
        "role": "user",
        "content": f"Extract the keyword profile from this job description:\n\n{test_jd}"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,
    top_p = 0.9
)

generated_output = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("Extracted Profile Output:\n", generated_output)
print("-" * 50)

print("Loading evaluation metrics suite (ROUGE, BLEU, BERTScore)...")
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")

test_data = eval_dataset

generated_predictions = []
ground_truth_references = []
llm_judge_payloads = []

print(f"Running batch inference on {len(test_data)} test split samples...")

with torch.no_grad():
    for sample in tqdm(test_data):
        sample_messages = sample.get('messages', [])

        if not sample_messages or len(sample_messages) < 2:
            continue

        ground_truth = sample_messages[-1]['content']

        prompt_messages = sample_messages[:-1]

        eval_inputs = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize = True,
            add_generation_prompt = True,
            return_tensors = "pt",
        ).to("cuda")

        eval_outputs = model.generate(
            input_ids = eval_inputs,
            max_new_tokens = 512,
            use_cache = True,
            do_sample = False,
            pad_token_id = tokenizer.eos_token_id
        )

        generated_response = tokenizer.decode(eval_outputs[0][len(eval_inputs[0]):], skip_special_tokens=True).strip()

        generated_predictions.append(generated_response)
        ground_truth_references.append(ground_truth)

        user_query = next((m['content'] for m in prompt_messages if m['role'] == 'user'), "N/A")

        llm_judge_payloads.append({
            "context": user_query,
            "ground_truth": ground_truth,
            "model_output": generated_response
        })

print("\n--- Computing Dataset Statistics ---")

rouge_results = rouge_metric.compute(
    predictions=generated_predictions,
    references=ground_truth_references
)

bleu_results = bleu_metric.compute(
    predictions=generated_predictions,
    references=[[ref] for ref in ground_truth_references]
)

bertscore_results = bertscore_metric.compute(
    predictions=generated_predictions,
    references=ground_truth_references,
    lang="en"
)
avg_bert_precision = sum(bertscore_results['precision']) / len(bertscore_results['precision'])
avg_bert_recall = sum(bertscore_results['recall']) / len(bertscore_results['recall'])
avg_bert_f1 = sum(bertscore_results['f1']) / len(bertscore_results['f1'])

print("\n" + "="*50)
print("             FINAL EVALUATION REPORT             ")
print("="*50)
print(f"1. ROUGE-1 (Keyword Match):      {rouge_results['rouge1']:.4f}")
print(f"2. ROUGE-2 (Phrase Match):       {rouge_results['rouge2']:.4f}")
print(f"3. ROUGE-L (Sentence Structure): {rouge_results['rougeL']:.4f}")
print("-"*50)
print(f"4. BLEU Score (Fluency/N-gram):  {bleu_results['bleu']:.4f}")
print("-"*50)
print(f"5. BERTScore Precision:          {avg_bert_precision:.4f}")
print(f"6. BERTScore Recall:             {avg_bert_recall:.4f}")
print(f"7. BERTScore F1 (Semantic Text): {avg_bert_f1:.4f}")
print("="*50)

if llm_judge_payloads:
    print("\n[INFO] Copy the template block below to run an 'LLM-as-a-Judge' audit on a sample output:")
    print("\n'''")
    print("You are an expert HR data auditing assistant. Rate the extracted model output out of 10 based on schema correctness, keyword extraction completeness, and lack of hallucination.")
    print(f"ORIGINAL JOB DESCRIPTION:\n{llm_judge_payloads[0]['context']}\n")
    print(f"GROUND TRUTH TARGET JSON:\n{llm_judge_payloads[0]['ground_truth']}\n")
    print(f"MODEL EXTRACTED JSON:\n{llm_judge_payloads[0]['model_output']}\n")
    print("Provide your breakdown rating and a final score from 1-10.")
    print("'''")

Loading fine-tuned Unsloth model for evaluation...
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running Quick Manual Verification ---
Extracted Profile Output:
 {"primary":["Senior DevOps Engineer","Kubernetes cluster administration","Terraform","AWS Infrastructure","Gitlab CI/CD","Prometheus","Grafana"],"secondary":[],"tertiary":[]}
--------------------------------------------------
Loading evaluation metrics suite (ROUGE, BLEU, BERTScore)...


Running batch inference on 251 test split samples...


  7%|▋         | 17/251 [04:54<1:04:35, 16.56s/it]Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
